In [1]:
cd "E:/AbbasCentral/python"

E:\AbbasCentral\python


In [2]:
import os
print(os.getcwd())


E:\AbbasCentral\python


In [3]:
from BehDat.core import *
from open_ephys.analysis import Session

In [4]:
testPath = Path("E:/Ephys/OpenEphys/testRecord")
#testPath = Path("E:/Ephys/OpenEphys/bpod_test")
#testPath = Path("E:/Ephys/OpenEphys/2026-07-07_17-54-35")
testSession = Session(testPath)

In [5]:
numSamples = testSession.recordnodes[0].recordings[0].continuous[0].samples.shape[0]
baud = testSession.recordnodes[0].recordings[0].continuous[0].metadata.sample_rate


In [6]:
from configparser import ConfigParser
import re
import numpy as np
iniPath = Path("E:/Ephys/OpenEphys/testTemplate.ini")
I = ConfigParser()
I.read(iniPath)
startState = I['info']['startstate']
regions = dict(I['regions'])
for region in regions.keys():
    vals = regions[region]
    cleanStr = vals.strip("[] ")
    if not cleanStr:
        regions[region] = np.array([], dtype=int)
        continue

    parts = cleanStr.split(",")
    result = []

    for part in parts:
        part = part.strip()
        if ":" in part:
            start, end = map(int, part.split(":"))
            result.extend(range(start, end+1))
        else:
            result.append(int(part))
    regions[region] = result

In [7]:
testInfo = BehDatInfo(acquisition=Acquisition.OPEN_EPHYS,
                      path=testPath,
                      name="testRecord",
                      baud=baud,
                      samples=numSamples,
                      trialTypes=dict(I['trialTypes']),
                      outcomes=dict(I['outcomes']),
                      stimTypes=dict(I['stimTypes']),
                      condition='test',
                      startState=startState,
                      channels=regions)

In [8]:
import numpy as np
import pandas as pd
print(testPath)
print('')
unsortedSpikeTimes = np.load(testPath / 'spike_times.npy')
unsortedSpikeClusters = np.load(testPath / 'spike_clusters.npy')
clusterInfo = pd.read_csv(testPath / 'cluster_info.tsv', sep='\t')

E:\Ephys\OpenEphys\testRecord



In [9]:
goodClusters = clusterInfo.loc[clusterInfo['KSLabel'] == 'good']
goodClusterID = goodClusters['cluster_id']
goodChannels = goodClusters['ch']
unitNo = 0
spikeInfo = {}
for clust in goodClusterID:
    clustRegion = ''
    for region in regions.keys():
        if np.any(np.array(regions[region]) == goodChannels[clust]):
            clustRegion = region
            break
    spikeInds = np.argwhere(unsortedSpikeClusters == clust)
    spikeTimes = unsortedSpikeTimes[spikeInds]
    spikeInfo[unitNo] = {'times': spikeTimes, 'region': clustRegion}
    unitNo += 1

In [10]:
print(testSession.recordnodes[0].recordings[0].continuous[0].metadata.channel_names)

['CH1', 'CH2', 'CH3', 'CH4', 'CH5', 'CH6', 'CH7', 'CH8', 'CH9', 'CH10', 'CH11', 'CH12', 'CH13', 'CH14', 'CH15', 'CH16', 'CH17', 'CH18', 'CH19', 'CH20', 'CH21', 'CH22', 'CH23', 'CH24', 'CH25', 'CH26', 'CH27', 'CH28', 'CH29', 'CH30', 'CH31', 'CH32', 'CH33', 'CH34', 'CH35', 'CH36', 'CH37', 'CH38', 'CH39', 'CH40', 'CH41', 'CH42', 'CH43', 'CH44', 'CH45', 'CH46', 'CH47', 'CH48', 'CH49', 'CH50', 'CH51', 'CH52', 'CH53', 'CH54', 'CH55', 'CH56', 'CH57', 'CH58', 'CH59', 'CH60', 'CH61', 'CH62', 'CH63', 'CH64']


In [11]:
timestampKeys = dict(I['timestamps'])

continuousRec = testSession.recordnodes[0].recordings[0].continuous[0]
print(continuousRec.samples.shape)
channelNames = continuousRec.metadata.channel_names
adcIdx = np.strings.find(channelNames, 'ADC') != -1
adcChannels = continuousRec.samples[:, adcIdx]
print(adcChannels)
"""
if ~isempty(adcChannels)
    bpodChannels = adcChannels(1:3);
    videoChannel = adcChannels(4);
    bpodData = ephysData.samples(bpodChannels, :);
    videoData = ephysData.samples(videoChannel, :);
    timestamps = get_bpod_analog_timestamps(bpodData);
    timestamps.keys = ini.timestamps;
else"""

(24242137, 64)
[]


'\nif ~isempty(adcChannels)\n    bpodChannels = adcChannels(1:3);\n    videoChannel = adcChannels(4);\n    bpodData = ephysData.samples(bpodChannels, :);\n    videoData = ephysData.samples(videoChannel, :);\n    timestamps = get_bpod_analog_timestamps(bpodData);\n    timestamps.keys = ini.timestamps;\nelse'

In [12]:
import matplotlib.pyplot as plt

In [13]:
# get bpod analog timestamps
"""
7 combinations of wire signaling: 1, 2, 3, 1+2, 1+3, 2+3, 1+2+3
binary: 
[0 0 1](1)
[0 1 0](2)
[0 1 1](1+2)
[1 0 0](3)
[1 0 1](1+3)
[1 1 0](2+3)
[1 1 1](1+2+3)
"""
if adcChannels.size == 0:
    print('no bpod')
    digitalTS = {}
else:
    tsTolerance = 40000   #30 # how far apart signals can be to be considered from the same call
    timestamps = []
    for i in range(0, 3):
        dataStream = np.array(adcChannels[:, i])
        frameDiff = np.diff(dataStream)
        timestamps.append(np.intersect1d(np.where(frameDiff > 1000), np.where(dataStream[:-1] < 300)))
    digitalTS = [[] for i in range(0, 7)]
    for i in [7, 6, 5, 3]:  # reverse ordered binary values that share channels
        binByte = np.unpackbits(np.array([i], dtype=np.uint8))
        binIdx = np.flip(binByte[-3:])
        numStreams = sum(binIdx)
        streamIdxs = [idx for idx, keep in enumerate(binIdx) if keep]
        currentTS = [timestamps[idx] for idx in streamIdxs]
        firstTS = currentTS[0]

        removeMasks = [np.zeros(len(ts), dtype=bool) for ts in timestamps]

        tsRange = [[-tsTolerance + ts, tsTolerance + ts] for ts in firstTS]
        for ts in firstTS:
            sharedTS = []
            withinTol = [np.abs(times - ts) < tsTolerance for times in currentTS]
            if all(w.any() for w in withinTol):
                for streamPos, w in zip(streamIdxs, withinTol):
                    removeMasks[streamPos] |= w
        digitalTS[i - 1].append(firstTS[removeMasks[streamIdxs[0]]])
        timestamps = [a[~m] for a, m in zip(timestamps, removeMasks)]
    for idx, i in enumerate([4, 2, 1]):
        digitalTS[i - 1].append(timestamps[idx])

no bpod


In [14]:
testBeh = BehDat(info=testInfo, 
                 spikes=spikeInfo, 
                 timestamps=digitalTS, 
                 bpod=[], 
                 coordinates=[])

In [17]:
testBeh.bin_spikes()

bin spikes
